Finding best model and hyper parameter tunning using GridSearchCV

In [1]:
from sklearn.datasets import load_iris
data = load_iris()

In [4]:
import pandas as pd 

df = pd.DataFrame(data.data,columns=data.feature_names)
df['flower'] = data.target
df['flower'] = df['flower'].apply(lambda x: data.target_names[x])
df.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),flower
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [5]:
from sklearn.model_selection import train_test_split 
X_train, X_test, y_train, y_test = train_test_split(data.data,data.target , test_size= 0.3)

In [9]:
from sklearn.svm import SVC 
model = SVC(kernel='rbf',C=30,gamma='auto')
model.fit(X_train,y_train)
model.score(X_test,y_test)

0.9555555555555556

Using K-Fold

In [10]:
from sklearn.model_selection import cross_val_score 

In [11]:
cross_val_score(SVC(kernel='linear' , C=10 , gamma='auto'), data.data , data.target , cv=5 )

array([1.        , 1.        , 0.9       , 0.96666667, 1.        ])

In [ ]:
cross_val_score(SVC(kernel='rbf' , C=10 , gamma='auto'), data.data , data.target , cv=5 )

array([0.96666667, 1.        , 0.96666667, 0.96666667, 1.        ])

In [13]:
cross_val_score(SVC(kernel='rbf' , C=20 , gamma='auto'), data.data , data.target , cv=5 )

array([0.96666667, 1.        , 0.9       , 0.96666667, 1.        ])

Using loop

In [19]:
import numpy as np

kernels = ['rbf' , 'linear']
C = [5, 10 , 20]
avg_score = {}

for kval in kernels:
    for cval in C:
        cv_score = cross_val_score(SVC(kernel=kval , C= cval , gamma='auto') , data.data , data.target , cv=5 )
        avg_score[kval + '_' + str(cval) ] = float(np.average(cv_score))
        
avg_score

{'rbf_5': 0.9800000000000001,
 'rbf_10': 0.9800000000000001,
 'rbf_20': 0.9666666666666668,
 'linear_5': 0.9800000000000001,
 'linear_10': 0.9733333333333334,
 'linear_20': 0.9666666666666666}

Using GridSearchCV

In [21]:
from sklearn.model_selection import GridSearchCV 

clf = GridSearchCV(SVC(gamma='auto') , {
    'C' : [5 , 10 , 20] ,
    'kernel' : ['rbf' , 'linear']
} , cv=5 , return_train_score=False)

clf.fit(data.data , data.target)
clf.cv_results_

{'mean_fit_time': array([0.00076532, 0.00039978, 0.00079989, 0.00081449, 0.00099978,
        0.00059986]),
 'std_fit_time': array([6.93715195e-04, 4.89628863e-04, 3.99949423e-04, 4.08269618e-04,
        1.74290341e-06, 4.89784698e-04]),
 'mean_score_time': array([0.00081234, 0.00059986, 0.00039992, 0.00039921, 0.00040197,
        0.0006001 ]),
 'std_score_time': array([0.00040651, 0.00048978, 0.0004898 , 0.00048893, 0.00049232,
        0.00048998]),
 'param_C': masked_array(data=[5, 5, 10, 10, 20, 20],
              mask=[False, False, False, False, False, False],
        fill_value=999999),
 'param_kernel': masked_array(data=['rbf', 'linear', 'rbf', 'linear', 'rbf', 'linear'],
              mask=[False, False, False, False, False, False],
        fill_value=np.str_('?'),
             dtype=object),
 'params': [{'C': 5, 'kernel': 'rbf'},
  {'C': 5, 'kernel': 'linear'},
  {'C': 10, 'kernel': 'rbf'},
  {'C': 10, 'kernel': 'linear'},
  {'C': 20, 'kernel': 'rbf'},
  {'C': 20, 'kernel': 'li

In [23]:
df = pd.DataFrame(clf.cv_results_)
df

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_kernel,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.000765,0.000694,0.000812,0.000407,5,rbf,"{'C': 5, 'kernel': 'rbf'}",0.966667,1.0,0.966667,0.966667,1.0,0.980000,0.016330,1
1,0.000400,0.000490,0.000600,0.000490,5,linear,"{'C': 5, 'kernel': 'linear'}",1.000000,1.0,0.933333,0.966667,1.0,0.980000,0.026667,1
2,0.000800,0.000400,0.000400,0.000490,10,rbf,"{'C': 10, 'kernel': 'rbf'}",0.966667,1.0,0.966667,0.966667,1.0,0.980000,0.016330,1
3,0.000814,0.000408,0.000399,0.000489,10,linear,"{'C': 10, 'kernel': 'linear'}",1.000000,1.0,0.900000,0.966667,1.0,0.973333,0.038873,4
4,0.001000,0.000002,0.000402,0.000492,20,rbf,"{'C': 20, 'kernel': 'rbf'}",0.966667,1.0,0.900000,0.966667,1.0,0.966667,0.036515,5
5,0.000600,0.000490,0.000600,0.000490,20,linear,"{'C': 20, 'kernel': 'linear'}",1.000000,1.0,0.900000,0.933333,1.0,0.966667,0.042164,6


In [ ]:
df[['param_C','param_kernel','mean_test_score']]

,param_C,param_kernel,mean_test_score
0,5,rbf,0.980000
1,5,linear,0.980000
2,10,rbf,0.980000
3,10,linear,0.973333
4,20,rbf,0.966667
5,20,linear,0.966667


In [25]:
clf.best_params_

{'C': 5, 'kernel': 'rbf'}

In [26]:
clf.best_score_

np.float64(0.9800000000000001)

RandomizedSearchCV

In [27]:
from sklearn.model_selection import RandomizedSearchCV 

In [32]:
rs = RandomizedSearchCV(SVC(gamma='auto'), {
        'C': [1,10,20],
        'kernel': ['rbf','linear']
}, cv=5 , return_train_score=False , n_iter= 5)


rs.fit(data.data , data.target)
pd.DataFrame(rs.cv_results_)[['param_C','param_kernel','mean_test_score']]

,param_C,param_kernel,mean_test_score
0,1,linear,0.980000
1,20,linear,0.966667
2,10,rbf,0.980000
3,20,rbf,0.966667
4,10,linear,0.973333


In [40]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaled = scaler.fit_transform(data.data)

In [49]:
from sklearn import svm
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

model_params = {
    'svm': {
        'model': svm.SVC(gamma='auto'),
        'params' : {
            'C': [1,10,20],
            'kernel': ['rbf','linear']
        }  
    },
    'random_forest': {
        'model': RandomForestClassifier(),
        'params' : {
            'n_estimators': [1,5,10]
        }
    },
    'logistic_regression' : {
        'model': LogisticRegression(),
        'params': {
            'C': [1,5,10]
        }
    }
}

In [50]:
scores = []

for model_name, mp in model_params.items():
    clf =  GridSearchCV(mp['model'], mp['params'], cv=5, return_train_score=False)
    clf.fit(scaled, data.target)
    scores.append({
        'model': model_name,
        'best_score': clf.best_score_,
        'best_params': clf.best_params_
    })
    
df = pd.DataFrame(scores,columns=['model','best_score','best_params'])
df

,model,best_score,best_params
0,svm,0.973333,"{'C': 10, 'kernel': 'rbf'}"
1,random_forest,0.946667,{'n_estimators': 5}
2,logistic_regression,0.973333,{'C': 10}
